# ICD-11 Indexing Pipeline on Google Colab

This notebook runs the ICD-11 indexing pipeline to create vector embeddings for medical condition descriptions. The pipeline processes ICD-11 data and stores embeddings in a ChromaDB vector database.

## Overview
- **Input**: ICD-11 descriptions JSON file
- **Output**: Vector database with embeddings  
- **Models**: Configurable sentence-transformer models
- **Storage**: ChromaDB with persistent storage

## 1. Install Required Packages

Install all the Python packages needed for the ICD-11 indexing pipeline.

In [2]:
# Install required packages
!pip install -q sentence-transformers langchain-huggingface langchain-core langchain-chroma chromadb matplotlib tqdm

# Check GPU availability
import torch
print(f"🔥 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎯 Device: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("💻 Using CPU")

🔥 CUDA available: True
🎯 Device: Tesla T4
💾 Memory: 14.7 GB


## 2. Clone or Upload Project Files

Option A: Clone from GitHub (if repository is public)  
Option B: Upload project files manually

In [3]:
# Option A: Clone repository (replace with your actual repo URL)
# !git clone https://github.com/your-username/rxassist-ai.git
# %cd rxassist-ai

# Option B: Create minimal project structure for manual upload
import os
import json

# Create project directories
os.makedirs("data/ICD-codes", exist_ok=True)
os.makedirs("src/indexing", exist_ok=True)
os.makedirs("scripts", exist_ok=True)
os.makedirs("dev", exist_ok=True)

print("📁 Project directories created!")
print("📋 Next step: Upload your files to the appropriate directories")
print("   - ICD-11 data → data/ICD-codes/icd11_vectordb_base.json")
print("   - Source code → src/indexing/indexing.py")
print("   - Script → scripts/run_indexing.py")
print("   - CUDA diagnostics → dev/diagnose_cuda.py")

📁 Project directories created!
📋 Next step: Upload your files to the appropriate directories
   - ICD-11 data → data/ICD-codes/icd11_vectordb_base.json
   - Source code → src/indexing/indexing.py
   - Script → scripts/run_indexing.py
   - CUDA diagnostics → dev/diagnose_cuda.py


In [4]:
# Option C: Mount Google Drive and copy files
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Copy files from Google Drive (adjust paths as needed)
!cp "/content/drive/MyDrive/rxassist-ai/data/ICD-codes/icd11_vectordb_base.json" "data/ICD-codes/"
!cp "/content/drive/MyDrive/rxassist-ai/src/indexing/indexing.py" "src/indexing/"
!cp "/content/drive/MyDrive/rxassist-ai/scripts/run_indexing.py" "scripts/"
!cp "/content/drive/MyDrive/rxassist-ai/dev/diagnose_cuda.py" "dev/"

print("💾 Google Drive mounted at /content/drive")
print("📋 Uncomment and modify the copy commands above to copy your files")

Mounted at /content/drive
💾 Google Drive mounted at /content/drive
📋 Uncomment and modify the copy commands above to copy your files


## 3. Set Up File Paths and Model Configuration

Configure the model and verify that all required files are in place.

In [5]:
# Configure model and paths
# MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"  # Default model
# MODEL_NAME = "jinaai/jina-embeddings-v3"  # Advanced model
MODEL_NAME = "jinaai/jina-embeddings-v2-base-en"  # Faster alternative

print(f"🤖 Selected model: {MODEL_NAME}")

# Verify required files exist
import os

required_files = [
    "data/ICD-codes/icd11_vectordb_base.json",
    "src/indexing/indexing.py",
    "scripts/run_indexing.py",
    "dev/diagnose_cuda.py"
]

print("\n📋 File verification:")
for file_path in required_files:
    if os.path.exists(file_path):
        file_size = os.path.getsize(file_path) / 1024  # KB
        print(f"   ✅ {file_path} ({file_size:.1f} KB)")
    else:
        print(f"   ❌ {file_path} - MISSING!")

# Show current directory structure
print("\n📁 Current directory structure:")
!find . -type f -name "*.py" -o -name "*.json" | head -10

🤖 Selected model: jinaai/jina-embeddings-v2-base-en

📋 File verification:
   ✅ data/ICD-codes/icd11_vectordb_base.json (38162.9 KB)
   ✅ src/indexing/indexing.py (7.4 KB)
   ✅ scripts/run_indexing.py (3.1 KB)
   ✅ dev/diagnose_cuda.py (1.2 KB)

📁 Current directory structure:
./.config/.last_update_check.json
./dev/diagnose_cuda.py
./data/ICD-codes/icd11_vectordb_base.json
./src/indexing/indexing.py
./scripts/run_indexing.py
./drive/MyDrive/IvanaSingh_internship/CirclesGame5/jsPsych-6.1.0/package-lock.json
./drive/MyDrive/IvanaSingh_internship/CirclesGame5/jsPsych-6.1.0/package.json
./drive/MyDrive/IvanaSingh_internship/CirclesGame5/analysis_code/00_read_vpstunden.py
./drive/MyDrive/IvanaSingh_internship/CirclesGame5/analysis_code/01_preprocess_data.py
./drive/MyDrive/IvanaSingh_internship/CirclesGame5_Ratings/code/01_items_to_jspsych.py


## 4. Run the Indexing Script

Execute the ICD-11 indexing pipeline to create vector embeddings.

In [ ]:
# Run the indexing script with progress monitoring
import subprocess
import sys
import threading
import time
from tqdm.notebook import tqdm
import re

print(f"🚀 Starting ICD-11 indexing with model: {MODEL_NAME}")
print("⏰ This may take 10-30 minutes depending on the model and data size...\n")

# Progress tracking variables
progress_bar = None
current_stage = "Initializing..."
total_documents = 0
processed_docs = 0

def parse_progress(line):
    """Parse progress information from script output"""
    global progress_bar, current_stage, total_documents, processed_docs

    # Look for total document count
    if "Loaded" in line and "ICD descriptions" in line:
        match = re.search(r'(\d+) ICD descriptions', line)
        if match:
            total_documents = int(match.group(1))
            print(f"📊 Total documents to process: {total_documents:,}")

    # Look for embedding progress
    if "Storing embeddings" in line:
        current_stage = "Creating embeddings..."
        if progress_bar is None and total_documents > 0:
            progress_bar = tqdm(total=total_documents, desc=current_stage, unit="docs")

    # Look for individual document processing
    if progress_bar and ("Document" in line or "doc" in line):
        processed_docs += 1
        progress_bar.update(1)
        progress_bar.set_description(f"{current_stage} ({processed_docs}/{total_documents})")

def run_with_progress():
    """Run the script and monitor progress"""
    global current_stage

    try:
        # Start the process
        process = subprocess.Popen([
            sys.executable, "scripts/run_indexing.py",
            "--model", MODEL_NAME
        ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
           universal_newlines=True, bufsize=1)

        # Read output line by line
        output_lines = []
        while True:
            line = process.stdout.readline()
            if not line and process.poll() is not None:
                break

            if line:
                line = line.strip()
                output_lines.append(line)
                print(line)  # Print each line as it comes
                parse_progress(line)

        # Wait for process to complete
        return_code = process.wait()

        if progress_bar:
            progress_bar.close()

        if return_code == 0:
            print("\n✅ Indexing completed successfully!")
        else:
            print(f"\n❌ Indexing failed with return code {return_code}")

        return return_code, output_lines

    except Exception as e:
        if progress_bar:
            progress_bar.close()
        print(f"❌ Unexpected error: {e}")
        return -1, []

# Execute with progress monitoring
return_code, output = run_with_progress()

print(f"\n📋 Process completed with return code: {return_code}")
if processed_docs > 0:
    print(f"📊 Processed {processed_docs:,} documents")

🚀 Starting ICD-11 indexing with model: jinaai/jina-embeddings-v2-base-en
⏰ This may take 10-30 minutes depending on the model and data size...

2025-06-30 09:06:13.165600: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751274373.194865    5272 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751274373.204086    5272 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-30 09:06:13.229562: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropria

Creating embeddings...:   0%|          | 0/35085 [00:00<?, ?docs/s]

Streaming output truncated to the last 5000 lines.
Storing embeddings for collection_icd_11:  45%|████▌     | 35326/77681 [40:57<40:47, 17.31it/s]


In [ ]:
# Alternative: Simple progress monitoring with periodic updates
import subprocess
import sys
import time
from IPython.display import display, clear_output
import threading

print(f"🚀 Starting ICD-11 indexing with model: {MODEL_NAME}")
print("⏰ Monitoring progress every 30 seconds...\n")

# Global variables for monitoring
process = None
start_time = time.time()

def monitor_progress():
    """Monitor the process and show periodic updates"""
    while process and process.poll() is None:
        elapsed = time.time() - start_time
        mins, secs = divmod(int(elapsed), 60)

        clear_output(wait=True)
        print(f"🚀 Starting ICD-11 indexing with model: {MODEL_NAME}")
        print("⏰ Monitoring progress every 30 seconds...\n")
        print(f"⏱️  Running for: {mins:02d}:{secs:02d}")
        print("🔄 Indexing in progress...")
        print("   - Loading ICD-11 data")
        print("   - Converting to document objects")
        print("   - Creating embeddings (this is the slow part)")
        print("   - Storing in ChromaDB")
        print("\n💡 Check the terminal output above for detailed progress")

        time.sleep(30)  # Update every 30 seconds

# Start the subprocess
try:
    process = subprocess.Popen([
        sys.executable, "scripts/run_indexing.py",
        "--model", MODEL_NAME
    ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)

    # Start monitoring in background
    monitor_thread = threading.Thread(target=monitor_progress)
    monitor_thread.daemon = True
    monitor_thread.start()

    # Wait for completion and capture output
    output, _ = process.communicate()

    # Final update
    elapsed = time.time() - start_time
    mins, secs = divmod(int(elapsed), 60)

    clear_output(wait=True)
    print(f"🚀 ICD-11 indexing with model: {MODEL_NAME}")
    print(f"⏱️  Total time: {mins:02d}:{secs:02d}")

    if process.returncode == 0:
        print("✅ Indexing completed successfully!")
    else:
        print(f"❌ Indexing failed with return code {process.returncode}")

    print("\n📋 Full output:")
    print("=" * 60)
    print(output)

except Exception as e:
    print(f"❌ Error running indexing: {e}")
    if process:
        process.terminate()

In [ ]:
# Verify results and show output directory
import os
import glob

# Generate expected output directory name
model_name_clean = MODEL_NAME.replace("/", "_").replace("-", "_")
output_dir = f"data/vector_db_{model_name_clean}"

print(f"📁 Expected output directory: {output_dir}")

if os.path.exists(output_dir):
    print("✅ Vector database created successfully!")

    # Show directory contents
    print(f"\n📋 Contents of {output_dir}:")
    for root, dirs, files in os.walk(output_dir):
        level = root.replace(output_dir, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 2 * (level + 1)
        for file in files:
            file_path = os.path.join(root, file)
            file_size = os.path.getsize(file_path) / 1024  # KB
            print(f"{subindent}{file} ({file_size:.1f} KB)")

    # Show statistics
    db_files = glob.glob(f"{output_dir}/**/*", recursive=True)
    total_size = sum(os.path.getsize(f) for f in db_files if os.path.isfile(f)) / (1024*1024)  # MB
    print(f"\n📊 Database statistics:")
    print(f"   Total files: {len([f for f in db_files if os.path.isfile(f)])}")
    print(f"   Total size: {total_size:.1f} MB")

else:
    print("❌ Vector database not found!")
    print("   Check the indexing output above for errors")

📁 Expected output directory: data/vector_db_jinaai_jina_embeddings_v3
✅ Vector database created successfully!

📋 Contents of data/vector_db_jinaai_jina_embeddings_v3:
vector_db_jinaai_jina_embeddings_v3/
  chroma_langchain_db/
    chroma.sqlite3 (286092.0 KB)
    3ed3fe29-9db5-40f1-a43d-a320bdf22804/
      header.bin (0.1 KB)
      link_lists.bin (642.7 KB)
      length.bin (300.8 KB)
      data_level0.bin (318527.3 KB)
      index_metadata.pickle (6918.4 KB)

📊 Database statistics:
   Total files: 6
   Total size: 598.1 MB


In [ ]:
# Download the vector database (optional)
from google.colab import files
import shutil

# Create a zip file of the vector database for download
if os.path.exists(output_dir):
    zip_filename = f"vector_db_{model_name_clean}.zip"

    print(f"📦 Creating zip file: {zip_filename}")
    shutil.make_archive(f"vector_db_{model_name_clean}", 'zip', output_dir)

    print(f"💾 Download ready: {zip_filename}")
    print("   Click the file in the Files panel to download")

    # Uncomment the line below to auto-download
    # files.download(zip_filename)

else:
    print("❌ No vector database to download")

print("\n🎉 ICD-11 indexing pipeline completed!")
print("📋 Summary:")
print(f"   Model used: {MODEL_NAME}")
print(f"   Output location: {output_dir}")
print(f"   Collection name: collection_icd_11")

📦 Creating zip file: vector_db_jinaai_jina_embeddings_v3.zip
💾 Download ready: vector_db_jinaai_jina_embeddings_v3.zip
   Click the file in the Files panel to download

🎉 ICD-11 indexing pipeline completed!
📋 Summary:
   Model used: jinaai/jina-embeddings-v3
   Output location: data/vector_db_jinaai_jina_embeddings_v3
   Collection name: collection_icd_11
